In [ ]:
#get parameters
import sys
sys.path.append('..')
from src.grass_functions import*
from project_info import *

#for project
Project_Area = 'nebraska_regression_stantec'
sr = '26852' #set to None if you want to use the DEM's original projection
res = '10m' #meters #DEM resolution, options are '1m', '3m', '10m', '30m', 'OPR'

#dem info turn into class later
dem_preprocessed = False
dem_base_name = 'state_dem' #for saving in grass
aligned = False
carved = True

#initiate class
project = ProjectInformation(Project_Area,sr,res,dem_preprocessed,dem_base_name,aligned,carved)
project.create_output_dirs()
initialize_grass_db(project.Location, project.Mapset, project.sr)
output_geo = []


for num in np.arange(2,76):#76
    #for run
    data_scale = 'pnt_id'
    analysis_scale = 'pnt_id'
    aoi = num
    geometry = 'point' #
    
    aoi = GrassWatershed(project, data_scale,analysis_scale,aoi,geometry)
    aoi.set_grass_selection()
    #initialize_grass_db(project.Location, project.Mapset, project.sr)
    aoi.list_existing_grass(print_it=False)
    aoi.set_dem_name()
    aoi.assign_grass_variables()
    aoi.get_grass_grid_size()
    aoi.get_rough_watershed_data(overwrite=True)
    aoi.get_basin_area()
    
    regression_data = RegressionData(project,aoi)
    regression_data.set_aoi()
    regression_data.get_drainage_area()
    regression_data.get_stream_order()
    regression_data.get_basin_shape()
    regression_data.get_basin_relief()
    regression_data.CN_mean = regression_data.get_raster_avg(project.raster_dir/'NE_State_CN_nad.tif','CN_mean')
    regression_data.PRISMyr_mm =regression_data.get_raster_avg(project.raster_dir/'PRISM_yr_NE_Statewide.tif','PRISMyr_mm')
    regression_data.get_main_ch_slope()
    gs.run_command('v.out.ogr', input=  aoi.v_basins ,type = 'area',output = aoi.basins, format = 'GeoJSON')
    
    #initialize
    regression_regions = project.vector_dir/'draft_regression.shp'
    regression_flows = RegressionEquations(project,aoi,regression_data,regression_regions)
    #regionalization
    regression_flows.get_regions()
    #calculate and add flows to geojson vector
    regression_flows.calc_flows()
    regression_flows.add_flows_to_outlet()
    output_geo.append(aoi.outlet)

/opt/conda/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.0
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Database Location Exists
Database Mapset Exists
None
{'GISDBASE': '/home/grassdata', 'LOCATION_NAME': 'nebraska_regression_stantec_26852', 'MAPSET': 'PERMANENT'}
base data is 2, analysis area is 2


Current GRASS GIS 7 environment:
/opt/conda/lib/python3.9/site-packages/geopandas/io/file.py:299: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,
Raster MASK removed


creating  project-wide watershed data for the project area


Beautify flat areas is not yet supported for disk swap mode
SECTION 1 beginning: Initiating Variables. 4 sections total.
SECTION 1a: Mark masked and NULL cells
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
SECTION 1b: Determining Offmap Flow.
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
SECTION 2: A* Search.
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
SECTION 3: Accumulating Surface Flow with SFD.
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  5

...

         overwritten
Buffering areas...
  50 100
Cleaning buffers...
Building parts of topology...
Building topology for vector map <aoi_2_aoi_raw_buffer@PERMANENT>...
Registering primitives...
Snapping boundaries...
Reading features...
Snap vertices Pass 1: select points
   0  50 100
Snap vertices Pass 2: assign anchor vertices
   4   9  14  19  24  29  34  39  44  49  54  59  64  69  74  79  84  89  94  99 100
Snap vertices Pass 3: snap to assigned points
   0  50 100
Breaking polygons...
Breaking polygons (pass 1: select break points)...
  50 100
Breaking polygons (pass 2: break at selected points)...
  50 100
Removing duplicates...
  50 100
Breaking boundaries...
   0  12  25  37  50  62  75  87 100
Removing duplicates...
  12  25  37  50  62  75  87 100
Cleaning boundaries at nodes
  16  33  50  66  83 100
Building topology for vector map <aoi_2_aoi_raw_buffer@PERMANENT>...
Building areas...
   0  12  25  37  50  62  75Removing dangles...
  83 100
Removing bridges...
  12  25  37 

Delineating the watersheds


Reading raster map <drain_dir_2>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Calculating basins using vector point map...
Delineating basins for 1 outlets...
   0 100
Writing raster map <r_basins_2>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100


Converting the delineated watershed rasters to vectors


Extracting areas...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Writing areas...
   0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68  72  76  80  84  88  92  96 100
Building topology for vector map <v_basins_2@PERMANENT>...
Registering primitives...
Building areas...
   0   4   9  13  18  22  27  31  36  40  45  50  54  59  63  68  72  77  81  86  90  95 100
Attaching islands...
   0 100
Attaching centroids...
   0  16  33  50  66  83 100
r.to.vect complete.
Exporting 12 areas (may take some time)...
   8  16  25  33  41  50  58  66  75  83  91 100
         category are written only when -c flag is given.
v.out.ogr complete. 6 features (Polygon type) written to <v_basins_2>
(GeoJSON format).
Raster MASK removed
All subsequent raster operations will be limited to the MASK area. Removing
or renaming raster map named 'MASK' will restore raster operations to
normal.
Reading